In [ ]:
!pip install transformers datasets trl accelerate

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOTrainer, GRPOConfig

# Load rollout data
with open("rollouts_sample.json") as f:
    data = json.load(f)

# Convert to dataset format
train_data = []

for rollout in data:
    train_data.append({
        "prompt": rollout["prompt"],
        "completion": rollout["completion"],
        "reward": rollout["final_score"]
    })

dataset = Dataset.from_list(train_data)

# Device handling
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load model
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
).to(device)

# GRPO config
config = GRPOConfig(
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    num_train_epochs=1,
    max_prompt_length=512,
    max_completion_length=128,
    logging_steps=1,
)

# Trainer
trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=config
)

# Train
trainer.train()

In [ ]:
trainer.save_model("/content/grpo_model")
print("Model saved to /content/grpo_model")